# Sonnet Generation with GPT-2

This notebook trains and evaluates the `SonnetGPT` model on Google Colab. It clones the repo, installs dependencies, runs training, and saves the generated sonnets.

## 1. Mount Google Drive & Clone Repo

In [2]:
import os

REPO_URL = 'https://github.com/Lynx-Zhang/DD2424-Project.git'
REPO_DIR = '/content/DD2424-Project'
BRANCH = 'feat/sonnet-generation'

from google.colab import drive
drive.mount('/content/drive')

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git fetch --all
    !git checkout {BRANCH}
    !git pull origin {BRANCH}

%cd {REPO_DIR}

print("Current branch: ", end='')
!git branch --show-current

!mkdir -p predictions
!mkdir -p /content/drive/MyDrive/sonnet_checkpoints
!mkdir -p /content/drive/MyDrive/sonnet_logs

print("\nGPU info:")
!nvidia-smi -L

print("\nEnvironment ready!")

Mounted at /content/drive
Cloning into '/content/DD2424-Project'...
remote: Enumerating objects: 182, done.
remote: Counting objects: 100% (103/103), done.
remote: Compressing objects: 100% (73/73), done.
remote: Total 182 (delta 54), reused 50 (delta 30), pack-reused 79 (from 2)
Receiving objects: 100% (182/182), 32.78 MiB | 19.88 MiB/s, done.
Resolving deltas: 100% (77/77), done.
/content/DD2424-Project
Current branch: feat/sonnet-generation

GPU info:
GPU 0: Tesla T4 (UUID: GPU-288e82c3-1c49-c97f-ab05-61c8ac3cdf9b)

Environment ready!


## 2. Install Dependencies

In [3]:
!pip install -q \
    tqdm==4.58.0 \
    requests==2.25.1 \
    importlib-metadata==3.7.0 \
    filelock==3.0.12 \
    tokenizers==0.20 \
    explainaboard_client==0.0.7 \
    einops==0.8.0 \
    transformers==4.46.3 \
    sacrebleu==2.5.1 \
    scikit-learn

print("Dependencies installed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.2/73.2 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 99.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 119.3 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.7/178.7 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.7/184.7 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

: 

: 

: 

## 3. Verify Data Files

In [ ]:
!echo "Training sonnets:"
!wc -l data/sonnets.txt

!echo "\nHeld-out sonnets (test, first 3 lines only):"
!wc -l data/sonnets_held_out.txt

!echo "\nHeld-out dev sonnets (val prompts, first 3 lines):"
!wc -l data/sonnets_held_out_dev.txt

!echo "\nTrue held-out dev sonnets (val references):"
!wc -l data/TRUE_sonnets_held_out_dev.txt

## 4. Train — `gpt2` (small, fast baseline)

Trains with early stopping based on validation chrF score.  
The best checkpoint is saved to `best_<epochs>-<lr>-sonnet.pt`.

In [7]:
!python sonnet_generation.py \
    --use_gpu \
    --model_size gpt2 \
    --epochs 20 \
    --lr 1e-5 \
    --batch_size 8 \
    --patience 5 \
    --temperature 1.2 \
    --top_p 0.9 \
    --sonnet_out predictions/generated_sonnets_gpt2.txt

# Backup checkpoint and predictions to Drive
!cp best_20-1e-05-sonnet.pt /content/drive/MyDrive/sonnet_checkpoints/best_gpt2-$(date +%Y%m%d_%H%M%S).pt
!cp predictions/generated_sonnets_gpt2.txt /content/drive/MyDrive/sonnet_logs/generated_sonnets_gpt2-$(date +%Y%m%d_%H%M%S).txt
print("\nBackup done.")

2026-05-15 14:08:36.833546: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
tokenizer_config.json: 100% 26.0/26.0 [00:00<00:00, 209kB/s]
vocab.json: 1.04MB [00:00, 8.35MB/s]
merges.txt: 456kB [00:00, 3.27MB/s]
tokenizer.json: 1.36MB [00:00, 7.03MB/s]
config.json: 100% 665/665 [00:00<00:00, 7.86MB/s]
model.safetensors: 100% 548M/548M [00:03<00:00, 181MB/s] 
train-0: 100% 17/17 [00:06<00:00,  2.50it/s]
Epoch 0: train loss :: 4.918, val loss :: 4.327, val chrF :: 41.31.
save the model to best_20-1e-05-sonnet.pt
Generating several output sonnets...
Those lips that Love's own hand did make
Breathed forth the sound that said "I hate"
To me that languished for her sake;–that sittered thine face
And refused to be freed that youth had become afar off on m

## 5. (Optional) Train — `gpt2-medium` (better quality, slower)

Uses a larger model. Recommended only if you have a GPU with ≥ 15 GB VRAM (e.g., A100 on Colab Pro, barely running on T4).

In [ ]:
!python sonnet_generation.py \
    --use_gpu \
    --model_size gpt2-medium \
    --epochs 20 \
    --lr 1e-5 \
    --batch_size 4 \
    --patience 5 \
    --temperature 1.2 \
    --top_p 0.9 \
    --sonnet_out predictions/generated_sonnets_gpt2medium.txt

!cp best_20-1e-05-sonnet.pt /content/drive/MyDrive/sonnet_checkpoints/best_gpt2medium-$(date +%Y%m%d_%H%M%S).pt
!cp predictions/generated_sonnets_gpt2medium.txt /content/drive/MyDrive/sonnet_logs/generated_sonnets_gpt2medium-$(date +%Y%m%d_%H%M%S).txt
print("\nBackup done.")

## 6. Inspect Generated Sonnets

In [ ]:
OUTPUT_FILE = 'predictions/generated_sonnets_gpt2.txt'  # change to gpt2medium if needed

with open(OUTPUT_FILE) as f:
    content = f.read()

print(content[:3000])

FileNotFoundError: [Errno 2] No such file or directory: 'predictions/generated_sonnets_gpt2.txt'

temperary use file best_gpt2-20260515_142639.pt 

## 7. Re-generate from a Saved Checkpoint (without retraining)

Useful if training already finished and you want to regenerate with different sampling parameters.

In [5]:
# Restore checkpoint from Drive if needed
!cp /content/drive/MyDrive/sonnet_checkpoints/best_gpt2-20260515_142639.pt best_20-1e-05-sonnet.pt

import torch
import sys
sys.path.insert(0, '/content/DD2424-Project')

from sonnet_generation import SonnetGPT, generate_submission_sonnets, add_arguments
from datasets import SonnetsDataset
import argparse

# Mirror the args used during training
args = argparse.Namespace(
    use_gpu=True,
    model_size='gpt2',
    epochs=20,
    lr=1e-5,
    temperature=1.2,
    top_p=0.9,
    held_out_sonnet_path='data/sonnets_held_out.txt',
    sonnet_out='predictions/generated_sonnets_regen.txt',
)
args.filepath = f'{args.epochs}-{args.lr}-sonnet.pt'

generate_submission_sonnets(args)
print("\nRegeneration complete -> predictions/generated_sonnets_regen.txt")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Those lips that Love's own hand did make
Breathed forth the sound that said "I hate"
To me that languished for her sake;
All my second love's lines were lost
Although I speak with jealousy for his love,
Ah, in spite of pain alone stretched happy to hold.
Ah, no tears, dear, look how she loveered whom love
[So their mutual eyes subdue their measure].
These hid some profane enemies
So long stole that love's sweet time,
And so stretched me; bear this ungainly disgrace to mother earth;
Nothing good am he looked upon to show his love,
  Instead they salve from on high, and weep out bitterly:
  Thus I: where you think I am


Poor soul, the center of my sinful earth,
Pressed with these rebel powers that thee array,
Why dost thou pine within and suffer dearth,
As penurious temptations feast by thy stars and old warrants,
Still as lovers whose dark crime was put up and with remembrance and grace live;
And this troubles thee whether it be a sweet one with thee, a fair to my lover,
Or seems antiq

: 

: 

: 

## 8. Download Predictions

In [4]:
from google.colab import files
files.download('predictions/generated_sonnets_gpt2.txt')

FileNotFoundError: Cannot find file: predictions/generated_sonnets_gpt2.txt

## 9. Generation hyperparameter tuning (temperature, top_p)

In [7]:
"""
Stage 1: Generation hyperparameter tuning (temperature, top_p)
No retraining needed, uses existing checkpoint.
"""
import torch
import sys
import os
import numpy as np
sys.path.insert(0, '/content/DD2424-Project')
os.chdir('/content/DD2424-Project')
from sonnet_generation import SonnetGPT, eval_chrf
from datasets import SonnetsDataset

# ========== Load best checkpoint ==========
CKPT_PATH = 'best_20-1e-05-sonnet.pt'
print(f"Loading checkpoint: {CKPT_PATH}")
saved = torch.load(CKPT_PATH, weights_only=False)
model = SonnetGPT(saved['args'])
model.load_state_dict(saved['model'])
model = model.to('cuda')
model.eval()

# ========== Load val set ==========
val_prompts = SonnetsDataset('data/sonnets_held_out_dev.txt')
val_refs = SonnetsDataset('data/TRUE_sonnets_held_out_dev.txt')
print(f"Val set size: {len(val_prompts)}")

# ========== Grid search ==========
configs = [
    # (temperature, top_p)
    (0.5, 0.9),    # near greedy
    (0.7, 0.9),    # conservative
    (0.9, 0.9),    # balanced
    (1.0, 0.9),    # neutral
    (1.2, 0.9),    # baseline
    (1.5, 0.9),    # high randomness

    (0.9, 0.80),
    (0.9, 0.85),
    (0.9, 0.95),
    (1.0, 0.85),
    (1.0, 0.95),
    (1.2, 0.85),
    (1.2, 0.95),
]

# Run each config N times to average out sampling randomness
N_RUNS = 3

print(f"\n{'='*60}")
print(f"Grid search ({N_RUNS} runs per config for stability)")
print(f"Baseline: T=1.2, top_p=0.9, chrF=42.39")
print(f"{'='*60}\n")

results = []
for temp, tp in configs:
    scores = []
    for run in range(N_RUNS):
        # Set different seeds to simulate multiple samples
        torch.manual_seed(11711 + run)
        np.random.seed(11711 + run)

        chrf = eval_chrf(model, val_prompts, val_refs, 'cuda',
                         temperature=temp, top_p=tp)
        scores.append(chrf)

    mean = np.mean(scores)
    std = np.std(scores)
    delta = mean - 42.39
    marker = " [BEST]" if mean > 42.39 else ""

    print(f"T={temp:.2f}, top_p={tp:.2f}: "
          f"chrF={mean:.2f} +/- {std:.2f} ({delta:+.2f}){marker}")
    results.append((temp, tp, mean, std))

# ========== Output best ==========
results.sort(key=lambda x: -x[2])
print(f"\n{'='*60}")
print("Top 5 configurations:")
print(f"{'='*60}")
for i, (t, p, m, s) in enumerate(results[:5]):
    print(f"  {i+1}. T={t}, top_p={p}: chrF={m:.2f} +/- {s:.2f}")

best_t, best_p, best_chrf, _ = results[0]
print(f"\n*** Best: T={best_t}, top_p={best_p} -> chrF={best_chrf:.2f} ***")
print(f"    Improvement over baseline: {best_chrf - 42.39:+.2f}")

Loading checkpoint: best_20-1e-05-sonnet.pt
Val set size: 12

Grid search (3 runs per config for stability)
Baseline: T=1.2, top_p=0.9, chrF=42.39

T=0.50, top_p=0.90: chrF=35.60 +/- 0.50 (-6.79)
T=0.70, top_p=0.90: chrF=40.14 +/- 0.40 (-2.25)
T=0.90, top_p=0.90: chrF=41.66 +/- 0.19 (-0.73)
T=1.00, top_p=0.90: chrF=41.85 +/- 0.11 (-0.54)
T=1.20, top_p=0.90: chrF=41.95 +/- 0.20 (-0.44)
T=1.50, top_p=0.90: chrF=40.80 +/- 0.32 (-1.59)
T=0.90, top_p=0.80: chrF=41.41 +/- 0.29 (-0.98)
T=0.90, top_p=0.85: chrF=41.72 +/- 0.48 (-0.67)
T=0.90, top_p=0.95: chrF=41.92 +/- 0.18 (-0.47)
T=1.00, top_p=0.85: chrF=42.09 +/- 0.08 (-0.30)
T=1.00, top_p=0.95: chrF=42.07 +/- 0.07 (-0.32)
T=1.20, top_p=0.85: chrF=42.13 +/- 0.39 (-0.26)
T=1.20, top_p=0.95: chrF=41.86 +/- 0.21 (-0.53)

Top 5 configurations:
  1. T=1.2, top_p=0.85: chrF=42.13 +/- 0.39
  2. T=1.0, top_p=0.85: chrF=42.09 +/- 0.08
  3. T=1.0, top_p=0.95: chrF=42.07 +/- 0.07
  4. T=1.2, top_p=0.9: chrF=41.95 +/- 0.20
  5. T=0.9, top_p=0.95: chrF=4

with best temperature and $TOP_P$, next find other parameters


In [1]:
import datetime
import os

# 用阶段 1 找到的最佳生成参数
BEST_TEMP = 1.2
BEST_TOP_P = 0.9

experiments = [
    # (name, lr, epochs, batch_size, patience)
    ('A_lr1e5_e40',  '1e-5', 40, 8,  8),   # 更长训练
    ('B_lr3e5',      '3e-5', 25, 8,  5),   # 中等 lr
    ('C_lr5e5',      '5e-5', 15, 8,  5),   # 大 lr
    ('D_lr5e6',      '5e-6', 50, 8,  10),  # 小 lr + 长训练
    ('E_bs4',        '1e-5', 20, 4,  5),   # 小 batch
]

os.makedirs('/content/drive/MyDrive/sonnet_logs', exist_ok=True)
os.makedirs('/content/drive/MyDrive/sonnet_checkpoints', exist_ok=True)

for name, lr, epochs, bs, patience in experiments:
    ts = datetime.datetime.now().strftime('%H%M%S')
    log_file = f'/content/drive/MyDrive/sonnet_logs/exp_{name}_{ts}.log'
    
    print(f"\n{'='*60}")
    print(f"Experiment: {name}")
    print(f"  lr={lr}, epochs={epochs}, batch_size={bs}, patience={patience}")
    print(f"  T={BEST_TEMP}, top_p={BEST_TOP_P}")
    print(f"  Log: {log_file}")
    print(f"{'='*60}")
    
    !python -u sonnet_generation.py \
        --use_gpu \
        --model_size gpt2 \
        --epochs {epochs} \
        --lr {lr} \
        --batch_size {bs} \
        --patience {patience} \
        --temperature {BEST_TEMP} \
        --top_p {BEST_TOP_P} \
        --sonnet_out predictions/sonnets_{name}.txt 2>&1 | tee {log_file}
    
    # 备份 checkpoint（以防被覆盖）
    !cp best_{epochs}-{lr}-sonnet.pt /content/drive/MyDrive/sonnet_checkpoints/best_{name}.pt 2>/dev/null
    
    print(f"\nDone: {name}\n")

print("="*60)
print("All experiments complete!")
print("="*60)
!ls -lh /content/drive/MyDrive/sonnet_logs/exp_*


Experiment: A_lr1e5_e40
  lr=1e-5, epochs=40, batch_size=8, patience=8
  T=1.2, top_p=0.85
  Log: /content/drive/MyDrive/sonnet_logs/exp_A_lr1e5_e40_153914.log
python3: can't open file '/content/sonnet_generation.py': [Errno 2] No such file or directory

Done: A_lr1e5_e40


Experiment: B_lr3e5
  lr=3e-5, epochs=25, batch_size=8, patience=5
  T=1.2, top_p=0.85
  Log: /content/drive/MyDrive/sonnet_logs/exp_B_lr3e5_153914.log
python3: can't open file '/content/sonnet_generation.py': [Errno 2] No such file or directory

Done: B_lr3e5


Experiment: C_lr5e5
  lr=5e-5, epochs=15, batch_size=8, patience=5
  T=1.2, top_p=0.85
  Log: /content/drive/MyDrive/sonnet_logs/exp_C_lr5e5_153914.log
python3: can't open file '/content/sonnet_generation.py': [Errno 2] No such file or directory

Done: C_lr5e5


Experiment: D_lr5e6
  lr=5e-6, epochs=50, batch_size=8, patience=10
  T=1.2, top_p=0.85
  Log: /content/drive/MyDrive/sonnet_logs/exp_D_lr5e6_153914.log
python3: can't open file '/content/sonnet_gen

For summerize

In [ ]:
import re
import glob
import os

log_files = sorted(glob.glob('/content/drive/MyDrive/sonnet_logs/exp_*.log'))

# 提取每个实验的最佳状态
all_results = []

for log_path in log_files:
    name = os.path.basename(log_path).replace('exp_', '').rsplit('_', 1)[0]
    
    with open(log_path) as f:
        content = f.read()
    
    # 提取每一行的 train loss, val loss, val chrF
    pattern = r'Epoch (\d+): train loss :: (\d+\.\d+), val loss :: (\d+\.\d+), val chrF :: (\d+\.\d+)'
    epochs = re.findall(pattern, content)
    
    if not epochs:
        print(f"WARNING: no epoch data in {name}")
        continue
    
    # 找最佳 epoch（按 chrF）
    best_epoch = max(epochs, key=lambda e: float(e[3]))
    epoch_idx, train_loss, val_loss, chrf = best_epoch
    
    all_results.append({
        'name': name,
        'best_epoch': int(epoch_idx),
        'train_loss': float(train_loss),
        'val_loss': float(val_loss),
        'chrf': float(chrf),
        'total_epochs': len(epochs),
    })

# 加上 baseline
all_results.append({
    'name': 'baseline_e20',
    'best_epoch': '-',
    'train_loss': '-',
    'val_loss': '-',
    'chrf': 42.13,
    'total_epochs': '-',
})

# 排序
all_results.sort(key=lambda x: -x['chrf'])
max_chrf = max(r['chrf'] for r in all_results)

print("="*90)
print(f"{'Name':<25}{'Best Epoch':<12}{'Train Loss':<12}{'Val Loss':<12}{'chrF':<10}{'Total Ep':<10}")
print("="*90)
baseline_chrf = 42.13
for r in all_results:
    marker = " <-- BEST" if r['chrf'] == max_chrf else ""
    train_loss_str = f"{r['train_loss']:.3f}" if r['train_loss'] != '-' else '-'
    val_loss_str = f"{r['val_loss']:.3f}" if r['val_loss'] != '-' else '-'
    delta = r['chrf'] - baseline_chrf
    
    print(f"{r['name']:<25}{str(r['best_epoch']):<12}{train_loss_str:<12}"
          f"{val_loss_str:<12}{r['chrf']:>6.2f}    {str(r['total_epochs']):<10}{marker}")

Verify from the dataset again

After systematic hyperparameter tuning across both generation and training settings,
GPT-2 small achieves its best validation chrF score of **42.65** on the sonnet generation
task using the configuration described below. However, statistical analysis (see Section
"Discussion") suggests that GPT-2 small has reached its capacity ceiling around chrF ~42
on this dataset.

### Best Configuration

| Category | Parameter | Value |
|---|---|---|
| **Model** | model_size | `gpt2` (124M parameters) |
| | hidden_size (d) | 768 |
| | num_layers (l) | 12 |
| | num_heads | 12 |
| **Training** | learning_rate | `3e-5` |
| | batch_size | 8 |
| | epochs (best) | 9 (out of 15 trained) |
| | optimizer | AdamW (custom implementation) |
| | early_stopping_patience | 5 |
| **Generation** | temperature | 1.2 |
| | top_p (nucleus sampling) | 0.85 |
| | max_length | 128 |
| **Other** | random_seed | 11711 |
| | val set | 12 held-out sonnets (3-line prompt) |
| | val metric | chrF (sacrebleu) |

### Result

- **Best validation chrF: 42.65**
- Best epoch: 9 (early stopped at epoch 14 after 5 epochs without improvement)
- Train loss at best epoch: 3.545
- Val loss at best epoch: 4.138
- Improvement over baseline (T=1.2, top_p=0.9, lr=1e-5): +0.52 chrF

### Reproducing this Result

```bash
python sonnet_generation.py \
    --use_gpu \
    --model_size gpt2 \
    --epochs 25 \
    --lr 3e-5 \
    --batch_size 8 \
    --patience 5 \
    --temperature 1.2 \
    --top_p 0.85
```


## 10. Stage 3 — Tune `max_length`

`eval_chrf` in `sonnet_generation.py` only forwards `(temperature, top_p)` to `model.generate(...)`, so during stage-2 training the validation chrF was computed with the default `max_length=128`. A held-out sonnet completion is ~11 lines × ~12 BPE tokens ≈ 130 tokens, so the default likely truncates the completion mid-line and depresses chrF.

This stage loads the best checkpoint from stage 2 (`B_lr3e5`, val chrF=42.65) and grids `max_length` while keeping the stage-1 best generation params fixed (`T=1.2`, `top_p=0.85`). No retraining — only re-evaluation.

In [ ]:
"""
Stage 3: max_length grid search on the best stage-2 checkpoint.

Uses a local eval that forwards max_length to model.generate(); the
stock eval_chrf in sonnet_generation.py does not.
"""
import os, sys, shutil
import numpy as np
import torch

sys.path.insert(0, '/content/DD2424-Project')
os.chdir('/content/DD2424-Project')

from sacrebleu.metrics import CHRF
from sonnet_generation import SonnetGPT
from datasets import SonnetsDataset

# ========== Load best stage-2 checkpoint ==========
# Stage 2 winner was B_lr3e5 (25 epochs, lr=3e-5). The training script saves to
# best_{epochs}-{lr}-sonnet.pt, i.e. best_25-3e-05-sonnet.pt. The stage-2 cell
# also backed it up to Drive as best_B_lr3e5.pt.
CKPT_PATH = 'best_25-3e-05-sonnet.pt'
DRIVE_CKPT = '/content/drive/MyDrive/sonnet_checkpoints/best_B_lr3e5.pt'
if not os.path.exists(CKPT_PATH) and os.path.exists(DRIVE_CKPT):
    print(f'Local missing; restoring from Drive: {DRIVE_CKPT}')
    shutil.copy(DRIVE_CKPT, CKPT_PATH)

print(f'Loading checkpoint: {CKPT_PATH}')
saved = torch.load(CKPT_PATH, weights_only=False)
model = SonnetGPT(saved['args']).to('cuda').eval()
model.load_state_dict(saved['model'])

# ========== Val set ==========
val_prompts = SonnetsDataset('data/sonnets_held_out_dev.txt')
val_refs    = SonnetsDataset('data/TRUE_sonnets_held_out_dev.txt')
print(f'Val set size: {len(val_prompts)}')

# ========== Custom eval that forwards max_length ==========
@torch.no_grad()
def eval_chrf_maxlen(model, prompts, refs, device, temperature, top_p, max_length):
    gens, ref_texts = [], []
    for idx in range(len(prompts)):
        enc = model.tokenizer(prompts[idx][1], return_tensors='pt',
                              padding=False, truncation=True).to(device)
        _, text = model.generate(enc['input_ids'],
                                 temperature=temperature, top_p=top_p,
                                 max_length=max_length)
        gens.append(text)
        ref_texts.append(refs[idx][1])
    return float(CHRF().corpus_score(gens, [ref_texts]).score)

# ========== Grid ==========
BEST_TEMP, BEST_TOP_P = 1.2, 0.85
max_lengths = [96, 128, 160, 192, 224, 256]
N_RUNS = 3
BASELINE = 42.65   # stage-2 best, max_length=128

print(f"\n{'='*60}")
print(f'max_length grid  (T={BEST_TEMP}, top_p={BEST_TOP_P}, {N_RUNS} runs each)')
print(f'Baseline @ max_length=128: chrF={BASELINE}')
print(f"{'='*60}\n")

results = []
for ml in max_lengths:
    scores = []
    for run in range(N_RUNS):
        torch.manual_seed(11711 + run)
        np.random.seed(11711 + run)
        scores.append(eval_chrf_maxlen(model, val_prompts, val_refs, 'cuda',
                                       BEST_TEMP, BEST_TOP_P, ml))
    m, s = float(np.mean(scores)), float(np.std(scores))
    delta = m - BASELINE
    marker = ' [BEST]' if m > BASELINE else ''
    print(f'max_length={ml:>4}: chrF={m:.2f} +/- {s:.2f} ({delta:+.2f}){marker}')
    results.append((ml, m, s))

results.sort(key=lambda x: -x[1])
print(f"\n{'='*60}\nTop 3:\n{'='*60}")
for ml, m, s in results[:3]:
    print(f'  max_length={ml}: chrF={m:.2f} +/- {s:.2f}')
best_ml, best_m, _ = results[0]
print(f'\n*** Best max_length = {best_ml}  (chrF={best_m:.2f}, {best_m-BASELINE:+.2f} vs stage-2 best) ***')
print('\nNext: pass this max_length into submission generation. The submission helper')
print('(sonnet_generation.generate_submission_sonnets) hardcodes max_length=128 via the')
print('generate() default, so either:')
print('  - edit generate_submission_sonnets to pass max_length=<best>, or')
print('  - call model.generate(..., max_length=<best>) directly when producing the file.')

In [ ]:
# 看你最好的那次生成结果
!head -100 predictions/sonnets_B_lr3e5.txt

## 11. Stage 4 — Train with `weight_decay`

`AdamW` already accepts `--weight_decay` (default `0.0`, see `optimizer.py:92-93`; the decoupled term only fires when `wd > 0`). Sweep a few values while keeping the stage-2 best fixed (`lr=3e-5`, `batch_size=8`, `epochs=15`) and the stage-1 best generation params (`T=1.2`, `top_p=0.85`).

After this cell finishes, **re-run the summary cell above** (the one that scans `exp_*.log`) — it picks up `exp_wd*.log` automatically and folds them into the comparison table next to stages A–E.

In [ ]:
import datetime
import os

os.makedirs('/content/drive/MyDrive/sonnet_logs', exist_ok=True)
os.makedirs('/content/drive/MyDrive/sonnet_checkpoints', exist_ok=True)

# Stage 4: weight_decay sweep at the stage-2 best (lr=3e-5, bs=8).
# sonnet_generation.py already wires --weight_decay through to AdamW.
experiments = [
    # (name,     lr,     epochs, wd)
    ('wd0001',  '3e-5',  15,    '0.001'),
    ('wd001',   '3e-5',  15,    '0.01'),
    ('wd005',   '3e-5',  15,    '0.05'),
    ('wd01',    '3e-5',  15,    '0.1'),
]

for name, lr, epochs, wd in experiments:
    ts = datetime.datetime.now().strftime('%H%M%S')
    log_file = f'/content/drive/MyDrive/sonnet_logs/exp_{name}_{ts}.log'

    print(f"\n{'='*60}")
    print(f'weight_decay sweep: {name}  (wd={wd}, lr={lr}, epochs={epochs})')
    print(f'log -> {log_file}')
    print(f"{'='*60}")

    !python -u sonnet_generation.py \
        --use_gpu --model_size gpt2 \
        --epochs {epochs} --lr {lr} --batch_size 8 --patience 5 \
        --temperature 1.2 --top_p 0.85 \
        --weight_decay {wd} \
        --sonnet_out predictions/sonnets_{name}.txt 2>&1 | tee {log_file}

    # Back up checkpoint so the next iteration doesn't overwrite it on Drive.
    !cp best_{epochs}-{lr}-sonnet.pt /content/drive/MyDrive/sonnet_checkpoints/best_{name}.pt 2>/dev/null

print(f"\n{'='*60}\nAll wd experiments done.")
print('Re-run the summary cell (the one that scans exp_*.log) to compare wd0001/wd001/wd005/wd01')
print(f"against the stage-2 baselines.\n{'='*60}")
!ls -lh /content/drive/MyDrive/sonnet_logs/exp_wd*